# VN-ALPR — Train trên Google Colab

Runtime → Change runtime type → **T4 GPU**. Chạy lần lượt từng ô.

1. Cài đặt  2. Dữ liệu  3. Train detector  4. Train CRNN  5. Đánh giá + ablation  6. Export + benchmark  7. Tải model về

In [ ]:
# 1. Cài đặt — đổi URL thành repo GitHub của bạn
!git clone https://github.com/<your-username>/vn-alpr.git
%cd vn-alpr
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dữ liệu

Cách A: tải dataset YOLO từ Roboflow Universe (đặt API key trong Colab Secrets tên `ROBOFLOW_API_KEY`).
Cách B: upload `data/raw` (images/, labels/, texts.csv, test_texts.csv) lên Google Drive rồi copy vào.

In [ ]:
# Cách A — Roboflow (điền workspace/project/version của dataset bạn chọn)
!pip install -q roboflow
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace('<workspace>').project('<project>').version(1).download('yolov8', location='data/roboflow')

# gộp train/valid/test của Roboflow vào data/raw để tự chia lại theo seed cố định
import shutil, pathlib
for split in ['train', 'valid', 'test']:
    for kind in ['images', 'labels']:
        src = pathlib.Path('data/roboflow') / split / kind
        dst = pathlib.Path('data/raw') / kind
        dst.mkdir(parents=True, exist_ok=True)
        if src.exists():
            for f in src.iterdir():
                shutil.copy2(f, dst / f.name)
print(len(list(pathlib.Path('data/raw/images').iterdir())), 'images')

In [ ]:
# Cách B — từ Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/vn-alpr-data/raw data/

# Chia train/val/test (ảnh tự chụp own_* luôn vào test)
!python scripts/split_dataset.py --src data/raw --dst data/processed/detection --test-prefix own_

In [ ]:
# 3. Train detector (~30-60 phút trên T4 tuỳ số ảnh)
!python scripts/train_detector.py --model yolo11n.pt --epochs 100 --batch 32
!python scripts/train_detector.py --eval models/detector/best.pt

In [ ]:
# 4a. Pre-train CRNN trên dữ liệu tổng hợp
!python scripts/gen_synthetic.py --n 50000 --out data/synthetic/ocr --seed 0
!python scripts/gen_synthetic.py --n 2000 --out data/synthetic/ocr_val --seed 1
!python scripts/train_crnn.py --config configs/crnn.yaml --epochs 20 --train-dirs data/synthetic/ocr --val-dirs data/synthetic/ocr_val

In [ ]:
# 4b. Fine-tune CRNN trên crop biển số thật (cần data/raw/texts.csv)
!python scripts/make_ocr_crops.py --detection data/processed/detection --texts data/raw/texts.csv --out data/processed/ocr
!python scripts/train_crnn.py --config configs/crnn.yaml --resume models/crnn/best.pt --epochs 30 --lr 0.0005
!python scripts/train_crnn.py --config configs/crnn.yaml --eval models/crnn/best.pt

In [ ]:
# 5. Đánh giá end-to-end + ablation (kết quả cộng dồn vào outputs/ablation.csv)
TEST = '--images data/processed/detection/images/test --gt data/raw/test_texts.csv'
!pip install -q -r requirements-paddle.txt
!python scripts/evaluate.py {TEST} --ocr paddle --no-postprocess --tag "YOLO + PaddleOCR"
!python scripts/evaluate.py {TEST} --ocr paddle --tag "YOLO + PaddleOCR + postprocess"
!python scripts/evaluate.py {TEST} --no-postprocess --tag "YOLO + CRNN"
!python scripts/evaluate.py {TEST} --tag "YOLO + CRNN + postprocess"
import pandas as pd; pd.read_csv('outputs/ablation.csv')

In [ ]:
# 6. Export + benchmark (cần 1 video demo, ví dụ data/demo.mp4)
!python scripts/export_models.py --detector models/detector/best.pt --crnn models/crnn/best.pt
!python scripts/benchmark.py --source data/demo.mp4 --n 300
!python scripts/benchmark.py --source data/demo.mp4 --n 300 --detector models/detector/best.onnx --crnn models/crnn/best.onnx
# TensorRT FP16:
!python scripts/export_models.py --detector models/detector/best.pt --format engine --half
!python scripts/benchmark.py --source data/demo.mp4 --n 300 --detector models/detector/best.engine --crnn models/crnn/best.onnx

In [ ]:
# 7. Tải model + kết quả về máy
!zip -r vn_alpr_models.zip models outputs runs/detector/*/results.png runs/detector/*/confusion_matrix.png
from google.colab import files; files.download('vn_alpr_models.zip')